# Scenario 01 — Client Happy Path

Full client-side lifecycle: publish a source, publish a task against it, watch
for attempts, then confirm a response and verify the task reaches `FINALIZED`.

**Role:** Client only (no provider/master operations in this notebook).

**Prerequisites:**
- Testnet client wallet with gas
- `CLIENT_PRIVATE_KEY` exported in environment
- At least one provider on testnet actively attempting tasks (otherwise steps 3–6 time out)

**Flow:**

```
Client
  │
  ├─ 1. publish_source        → Source ACTIVE
  ├─ 2. publish_task           → Task NEW
  ├─ 3. watch_attempted        → wait for provider
  ├─ 4. task.get_responses()   → inspect responses
  ├─ 5. response.confirm()     → tx receipt
  └─ 6. task.get_status()      → FINALIZED
```


## 0. Setup


In [ ]:
import os
import time

from web3 import Web3

from ogpu import ChainConfig, ChainId
from ogpu.client import (
    DeliveryMethod,
    ImageEnvironments,
    SourceInfo,
    TaskInfo,
    TaskInput,
    publish_source,
    publish_task,
)
from ogpu.protocol import Master, Provider, Response, Source, Task, vault

ChainConfig.set_chain(ChainId.OGPU_TESTNET)

CLIENT_KEY = os.environ["CLIENT_PRIVATE_KEY"]


## 1. Publish a Source

`publish_source` returns a live `Source` instance bound to the freshly deployed contract.
Every method on the instance hits the chain on each call — no caching.


In [ ]:
source_info = SourceInfo(
    name="happy-path-demo",
    description="Scenario 01 demo source",
    logoUrl="https://example.com/logo.png",
    imageEnvs=ImageEnvironments(
        cpu="https://cipfs.ogpuscan.io/ipfs/QmNWFLL13ujf3KUTJvfNx42bA5fWDV96qqUdjY6nwpuwD9",
    ),
    minPayment=Web3.to_wei(0.01, "ether"),
    minAvailableLockup=Web3.to_wei(0, "ether"),
    maxExpiryDuration=86400,
    deliveryMethod=DeliveryMethod.FIRST_RESPONSE,
)

source = publish_source(source_info, private_key=CLIENT_KEY)
print(f"Source: {source.address}")
print(f"Status: {source.get_status()}")
print(f"Client: {source.get_client()}")


## 2. Publish a Task against the source

`publish_task` returns a `Task` instance. `task.get_source()` navigates back — proving the link.


In [ ]:
task_input = TaskInput(
    function_name="some_function",
    data={"prompt": "scenario 01 demo payload"},
)

task_info = TaskInfo(
    source=source.address,
    config=task_input,
    expiryTime=int(time.time()) + 3600,
    payment=Web3.to_wei(0.01, "ether"),
)

task = publish_task(task_info, private_key=CLIENT_KEY)
print(f"Task: {task.address}")
print(f"Status: {task.get_status()}")
print(f"Source (navigated): {task.get_source().address}")
print(f"Payment: {task.get_payment()} wei")


## 3. Watch for an attempt (async)

`ogpu.events` is the one async island in the SDK. `watch_attempted` is an async
generator that polls Nexus filter logs and yields typed events.


In [ ]:
import asyncio
from ogpu.events import watch_attempted

async def wait_for_first_attempt(task_addr: str, timeout: float = 120.0):
    async def inner():
        async for event in watch_attempted(task_addr, poll_interval=2.0):
            return event
    return await asyncio.wait_for(inner(), timeout=timeout)

attempt = await wait_for_first_attempt(task.address)
print(f"Attempt by: {attempt.provider}")
print(f"Suggested payment: {attempt.suggested_payment} wei")
print(f"Block: {attempt.block_number}")


## 4. Inspect submitted responses

After an attempt the provider eventually submits a `Response`. Fetch the current list.


In [ ]:
responses = task.get_responses()
print(f"{len(responses)} response(s) so far")
for r in responses:
    params = r.get_params()
    print(f"  {r.address}  provider={params.provider}  payment={params.payment}")
    print(f"  status={r.get_status()}  confirmed={r.is_confirmed()}")


## 5. Confirm the first response

Delivery method `FIRST_RESPONSE` still requires the client to call `confirm()` for the
`FINALIZED` status to land on-chain (status transitions are protocol-driven).


In [ ]:
target = responses[0]
receipt = target.confirm(signer=CLIENT_KEY)
print(f"Confirm tx: {receipt.tx_hash}")
print(f"Block: {receipt.block_number}  Gas: {receipt.gas_used}")


## 6. Verify the task reached FINALIZED


In [ ]:
print(f"Task status: {task.get_status()}")
print(f"Winning provider: {task.get_winning_provider()}")

# Snapshot captures all fields in one logical batch
snap = task.snapshot()
print(f"Snapshot: {snap}")
